In [13]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

## Random Split

In [2]:
msl_new = pd.read_csv('../input_data/msl_new.csv')
print(len(msl_new))
msl_new.head()

16648


,Chromophore,Solvent,Quantum yield
0,O=C([O-])c1ccccc1-c1c2ccc(=O)cc-2oc2cc([O-])ccc12,O,0.950
1,O=C([O-])c1ccccc1C1=c2cc3c4c(c2Oc2c1cc1c5c2CCC...,CO,1.000
2,O=C([O-])c1ccccc1-c1c2cc(Br)c(=O)c(Br)c-2oc2c(...,O,0.200
3,O=C([O-])c1ccccc1-c1c2cc(I)c(=O)c(I)c-2oc2c(I)...,O,0.020
4,O=C([O-])c1c(Cl)c(Cl)c(Cl)c(Cl)c1-c1c2cc(I)c(=...,O,0.018


In [3]:
vecs_mols = torch.load('../torch_data/mols_new.pt')
vecs_sols = torch.load('../torch_data/sols_new.pt')

C:\Users\kappa\AppData\Local\Temp\ipykernel_34212\4242151166.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vecs_mols = torch.load('../torch_data/mols_new.pt')
C:\Users

In [4]:
fgp_mols = torch.load('../torch_data/fingerprints.pt')
fgp_sols = torch.load('../torch_data/fingerprints_sol.pt')
print(len(fgp_mols), len(fgp_sols))
print(fgp_mols.shape, fgp_sols.shape)

C:\Users\kappa\AppData\Local\Temp\ipykernel_34212\42225485.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  fgp_mols = torch.load('../torch_data/fingerprints.pt')
C:\User

16648 16648
torch.Size([16648, 2048]) torch.Size([16648, 2048])


In [5]:
desc_mols = torch.load("../torch_data/descriptors.pt")
desc_sols = torch.load("../torch_data/descriptors_sols.pt")

C:\Users\kappa\AppData\Local\Temp\ipykernel_34212\3135145622.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  desc_mols = torch.load("../torch_data/descriptors.pt")
C:\Us

#### Fingerprints Only

In [6]:
Xf = torch.cat([fgp_mols, fgp_sols], dim=1)

print(Xf.shape)

yf = torch.tensor(msl_new['Quantum yield'].values)
print(yf.shape)

torch.Size([16648, 4096])
torch.Size([16648])


In [7]:
X_trainf, X_tempf, y_trainf, y_tempf = train_test_split(Xf, yf, test_size=0.2, random_state=42)
X_valf, X_testf, y_valf, y_testf = train_test_split(X_tempf, y_tempf, test_size=0.5, random_state=42)
print(X_trainf.shape, y_trainf.shape)
print(X_valf.shape, y_valf.shape)
print(X_testf.shape, y_testf.shape)

torch.Size([13318, 4096]) torch.Size([13318])
torch.Size([1665, 4096]) torch.Size([1665])
torch.Size([1665, 4096]) torch.Size([1665])


In [8]:
xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=4,
    reg_lambda=2.0,
    reg_alpha=0.2,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    tree_method="hist",
    device="cuda"
)

eval_set = [(X_valf, y_valf)]

xgb_model.fit(X_trainf, y_trainf, eval_set=eval_set, verbose=False)

y_test_xgb = xgb_model.predict(X_testf)
print("test:")
print(f"R Squared: {r2_score(y_testf, y_test_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_testf, y_test_xgb)):.3f}")

test:
R Squared: 0.661
RMSE: 0.176


### All Input Data

In [9]:
X = torch.cat([vecs_mols, vecs_sols, desc_mols, desc_sols, fgp_mols, fgp_sols], dim=1)

print(X.shape)

y = torch.tensor(msl_new['Quantum yield'].values)
print(y.shape)

torch.Size([16648, 5042])
torch.Size([16648])


In [10]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

torch.Size([13318, 5042]) torch.Size([13318])
torch.Size([1665, 5042]) torch.Size([1665])
torch.Size([1665, 5042]) torch.Size([1665])


#### XGB

In [34]:
xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=4,
    reg_lambda=2.0,
    reg_alpha=0.2,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    tree_method="hist",
    device="cuda"
)

eval_set = [(X_val, y_val)]

xgb_model.fit(X_train, y_train, eval_set=eval_set, verbose=False)

y_test_xgb = xgb_model.predict(X_test)
print("test:")
print(f"R Squared: {r2_score(y_test, y_test_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_xgb)):.3f}")

test:
R Squared: 0.713
RMSE: 0.162


In [35]:
'''
torch.save(X_train, "../torch_data/random_split/train/X_train.pt")
torch.save(y_train, "../torch_data/random_split/train/y_train.pt")

torch.save(X_val, "../torch_data/random_split/val/X_val.pt")
torch.save(y_val, "../torch_data/random_split/val/y_val.pt")

torch.save(X_test, "../torch_data/random_split/test/X_test.pt")
torch.save(X_test, "../torch_data/random_split/test/y_test.pt")
'''

'\ntorch.save(X_train, "../torch_data/random_split/train/X_train.pt")\ntorch.save(y_train, "../torch_data/random_split/train/y_train.pt")\n\ntorch.save(X_val, "../torch_data/random_split/val/X_val.pt")\ntorch.save(y_val, "../torch_data/random_split/val/y_val.pt")\n\ntorch.save(X_test, "../torch_data/random_split/test/X_test.pt")\ntorch.save(X_test, "../torch_data/random_split/test/y_test.pt")\n'

#### Random Forest

In [36]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    oob_score=True,
    random_state=42,
)

In [38]:
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
print("test:")
print(f"R Squared: {r2_score(y_test, y_pred_rf):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.3f}")

test:
R Squared: 0.662
RMSE: 0.175


#### Extra Trees

In [11]:
et_model = ExtraTreesRegressor(
    n_estimators=100,
    random_state=42
)

In [12]:
et_model.fit(X_train, y_train)

y_pred_et = et_model.predict(X_test)
print("test:")
print(f"R Squared: {r2_score(y_test, y_pred_et):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_et)):.3f}")

test:
R Squared: 0.722
RMSE: 0.159


#### MLP

In [41]:
class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dims=(512, 256, 64), dropout=0.2):
        super().__init__()

        layers = []
        prev_dim = input_dim

        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h

        layers.append(nn.Linear(prev_dim, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x).squeeze(-1)

In [42]:
print(X_train.shape[1])
device = torch.device("cuda")

5042


In [55]:
model = MLPRegressor(input_dim=5042, hidden_dims=(512, 256, 64), dropout=0.3).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

In [44]:
imp_m = SimpleImputer(strategy="median")
imp_s = SimpleImputer(strategy="median")

desc_mols_np = imp_m.fit_transform(desc_mols)
desc_sols_np = imp_s.fit_transform(desc_sols)

desc_mols_pt = torch.from_numpy(desc_mols_np)
desc_sols_pt = torch.from_numpy(desc_sols_np)

In [45]:
X = torch.cat([vecs_mols, vecs_sols, desc_mols_pt, desc_sols_pt, fgp_mols, fgp_sols], dim=1)
print(X.shape)

y = torch.tensor(msl_new['Quantum yield'].values)
print(y.shape)

torch.Size([16648, 5042])
torch.Size([16648])


In [46]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

X_train = X_train.float()
X_val = X_val.float()
X_test = X_test.float()

y_train = y_train.float()
y_val = y_val.float()
y_test = y_test.float()

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

torch.Size([13318, 5042]) torch.Size([13318])
torch.Size([1665, 5042]) torch.Size([1665])
torch.Size([1665, 5042]) torch.Size([1665])


In [47]:
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

In [48]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [56]:
for epoch in range(100):
    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device).float()

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device).float()

            preds = model(xb)
            loss = criterion(preds, yb)
            val_loss += loss.item() * xb.size(0)

    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}, Train {train_loss:.4f}, Val {val_loss:.4f}")

Epoch 1, Train 0.0724, Val 0.0588
Epoch 2, Train 0.0513, Val 0.0538
Epoch 3, Train 0.0422, Val 0.0455
Epoch 4, Train 0.0373, Val 0.0466
Epoch 5, Train 0.0331, Val 0.0429
Epoch 6, Train 0.0310, Val 0.0433
Epoch 7, Train 0.0291, Val 0.0405
Epoch 8, Train 0.0281, Val 0.0409
Epoch 9, Train 0.0270, Val 0.0401
Epoch 10, Train 0.0259, Val 0.0404
Epoch 11, Train 0.0247, Val 0.0394
Epoch 12, Train 0.0250, Val 0.0382
Epoch 13, Train 0.0240, Val 0.0385
Epoch 14, Train 0.0236, Val 0.0376
Epoch 15, Train 0.0234, Val 0.0386
Epoch 16, Train 0.0231, Val 0.0382
Epoch 17, Train 0.0226, Val 0.0400
Epoch 18, Train 0.0231, Val 0.0407
Epoch 19, Train 0.0221, Val 0.0385
Epoch 20, Train 0.0231, Val 0.0384
Epoch 21, Train 0.0217, Val 0.0382
Epoch 22, Train 0.0214, Val 0.0393
Epoch 23, Train 0.0212, Val 0.0390
Epoch 24, Train 0.0219, Val 0.0390
Epoch 25, Train 0.0207, Val 0.0369
Epoch 26, Train 0.0208, Val 0.0369
Epoch 27, Train 0.0205, Val 0.0366
Epoch 28, Train 0.0203, Val 0.0379
Epoch 29, Train 0.0203, Val 0

In [57]:
model.eval()

y_true = []
y_pred_mlp = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        yb = yb.to(device).float()

        preds = model(xb)

        y_true.append(yb.cpu().numpy())
        y_pred_mlp.append(preds.cpu().numpy())

y_true = np.concatenate(y_true)
y_pred_mlp = np.concatenate(y_pred_mlp)

print("test:")
print(f"R Squared: {r2_score(y_true, y_pred_mlp):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_true, y_pred_mlp)):.3f}")

test:
R Squared: 0.671
RMSE: 0.173
